In [2]:
import sys
sys.path.insert(0, '../onitama')

from dl_players_v7 import DensePlayer_v7
from alphazero import AlphaZeroTrainer

# ── Instanciation des joueurs ──────────────────────────────────────────────
p1 = DensePlayer_v7()
p2 = DensePlayer_v7()

p1.load_weights('../saved-models/Tairanauchu3.weights.h5')

# ── Paramètres volontairement très faibles pour test de non-régression ─────
trainer = AlphaZeroTrainer(
    player_p1=p1,
    player_p2=p2,
    n_games=2,           # 2 parties de self-play par itération
    n_simulations=10,    # 10 simulations MCTS par coup (au lieu de 100+)
    temperature_moves=5,
    n_epochs=1,
    minibatch_size=16,   # petit batch
    replay_buffer_size=500,
    learning_rate=1e-3,
)

# ── Lancement ─────────────────────────────────────────────────────────────
history = trainer.train(n_iterations=3)

# ── Vérifications post-run ─────────────────────────────────────────────────
print("=== Vérifications ===")
print(f"Buffer rempli       : {history['buffer_size']}")
print(f"Wins/Losses/Draws   : {list(zip(history['wins'], history['losses'], history['draws']))}")
print(f"Policy losses       : {history['policy_loss']}")
print(f"Value losses        : {history['value_loss']}")

# Ce qu'on vérifie :
# 1. buffer_size[-1] > 0            → des transitions ont bien été stockées
# 2. au moins un loss non-None      → _update() s'est exécuté
# 3. pas d'exception                → toute la plomberie fonctionne
assert history['buffer_size'][-1] > 0, "Buffer vide !"
losses = [l for l in history['policy_loss'] if l is not None]
assert len(losses) > 0, "Aucune mise à jour effectuée !"
assert all(l > 0 for l in losses), "Policy loss devrait être positive (cross-entropy)"
print(f"  → policy_loss moyen : {sum(losses)/len(losses):.4f}  (attendu : 5-8 en début d'entraînement)")
print("\nTout OK ✓")

Dégelé 4 layers de la tête de valeur
Dégelé 14 layers du tronc
Toutes les couches dégelées pour RL


Self-play: 100%|██████████| 2/2 [00:00<00:00,  2.96it/s]


[   1/3] buffer=44 | W/L/D=1/1/0 | p_loss=20.558155059814453 | v_loss=1.462216854095459


Self-play: 100%|██████████| 2/2 [00:00<00:00,  2.13it/s]


[   2/3] buffer=116 | W/L/D=1/1/0 | p_loss=23.204273223876953 | v_loss=1.1026649475097656


Self-play: 100%|██████████| 2/2 [00:00<00:00,  3.12it/s]

[   3/3] buffer=165 | W/L/D=1/1/0 | p_loss=22.02975845336914 | v_loss=1.0770927667617798
=== Vérifications ===
Buffer rempli       : [44, 116, 165]
Wins/Losses/Draws   : [(1, 1, 0), (1, 1, 0), (1, 1, 0)]
Policy losses       : [np.float64(20.558155059814453), np.float64(23.204273223876953), np.float64(22.02975845336914)]
Value losses        : [np.float64(1.462216854095459), np.float64(1.1026649475097656), np.float64(1.0770927667617798)]
  → policy_loss moyen : 21.9307  (attendu : 5-8 en début d'entraînement)

Tout OK ✓
